# Build the market, inspect it, and explain a valuation change

Market quotes are observations; curves and volatility surfaces are constructed
objects. This notebook connects the two and then changes one family of inputs
at a time. We compare a receive-fixed swap with a caplet: both depend on rates,
while only the caplet needs volatility. [Setup and reading order](../README.md).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import quantsupport as qs
from market_data import load_market

market = load_market()
ref = market.quotes.reference_date


## Construction specifications tell the context what to build

Here is the actual SOFR bootstrap specification loaded by the helper. Its
`quotes` list identifies the deposit/OIS instruments that constrain the curve.
The interpolation and day-count conventions govern values between pillars.
`from_dict` also lets you construct an editable configuration in memory;
`from_json` is useful for larger input sets.

The surface specification identifies caplet Black-volatility quotes. We keep
the long list of market identifiers in the auxiliary data and use typed enums
for the conventions we want to highlight.

In [ ]:
curve_spec = market.curves[0].to_dict()
curve_config = qs.CurveConfiguration.from_dict(curve_spec)
surface_spec = market.volatility_surfaces[0].to_dict()
surface_config = qs.VolatilitySurfaceConfiguration.from_dict(
    {
        **surface_spec,
        "market_index": qs.MarketIndex.SOFR,
        "volatility_type": qs.VolatilityType.Black,
        "smile_type": qs.SmileType.Strike,
    }
)
print(
    "Curve conventions:",
    {key: value for key, value in curve_spec.items() if key != "quotes"},
)
print(
    "Surface conventions:",
    {key: value for key, value in surface_spec.items() if key != "quotes"},
)
print(
    "Curve quotes:",
    len(curve_spec["quotes"]),
    "| Volatility quotes:",
    len(surface_spec["quotes"]),
)


## Observe nodes, forwards, and the smile

Query the constructed objects inside an active context. `nodes()` returns
dated discount factors; `pillars()` returns labels used to trace risk back to
inputs. A forward rate is inferred from discount factors over the requested
interval under the selected compounding convention.

This surface's smile coordinate is an **absolute strike**, so `0.045` means a
4.5% strike, not a delta or log-moneyness. Its values are decimal Black vols.
The grid below stays within the supplied quote domain and includes an
intermediate strike so the interpolation is visible.

In [ ]:
expiries = ["6M", "1Y", "2Y", "3Y", "5Y", "7Y"]
strikes = [0.035, 0.040, 0.045, 0.050, 0.055]
context = qs.PricingContext(
    quotes=market.quotes,
    curves=[curve_config],
    csa_policy=market.csa_policy,
    volatility_surfaces=[surface_config],
)

with context:
    curve = context.curve(qs.MarketIndex.SOFR)
    surface = context.volatility_surface(qs.MarketIndex.SOFR)
    nodes = curve.nodes()
    pillars = curve.pillars()
    forwards = pd.DataFrame(
        [
            {
                "start": tenor,
                "discount_factor": curve.discount_factor(ref + tenor),
                "one_year_forward": curve.forward_rate(
                    ref + tenor,
                    ref + tenor + "1Y",
                    qs.Compounding.Simple,
                    qs.Frequency.Annual,
                ),
            }
            for tenor in ["1Y", "2Y", "3Y", "5Y", "7Y"]
        ]
    )
    smile = pd.DataFrame(
        {
            expiry: [
                surface.volatility(
                    expiry,
                    strike,
                )
                for strike in strikes
            ]
            for expiry in expiries
        },
        index=pd.Index(
            strikes,
            name="strike",
        ),
    )
    print(
        surface.volatility_type,
        surface.smile_type,
    )

forwards


In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4),
)
axes[0].plot(
    pd.to_datetime(nodes["date"]),
    nodes["discount_factor"],
    marker="o",
)
axes[0].set(
    title="Constructed SOFR curve",
    xlabel="Maturity",
    ylabel="Discount factor",
)
(smile * 100).plot(
    ax=axes[1],
    marker="o",
)
axes[1].set(
    title="Caplet Black-volatility smiles",
    xlabel="Strike (decimal rate)",
    ylabel="Black volatility (%)",
)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


## Revalue the same contracts under changed inputs

A `Scenario` shocks quote values **before** construction. The context rebuilds
the dependent curve or surface; it does not add a bump directly to each
discount factor. This distinction preserves the meaning of a market-quote
stress.

Scenario selectors match underscore-delimited identifier segments. `SOFR`
would match both OIS rates and SOFR caplet vols in this snapshot. We therefore
use the exact curve quote IDs for a parallel rate-quote bump, and the surface
quote IDs for a volatility bump. `Absolute` adds the number; `Relative` would
multiply by `1 + shock`.

In [ ]:
swap = qs.Swap(
    identifier="SWAP",
    start_date=ref,
    maturity_date=ref + "5Y",
    notional=1_000_000,
    fixed_rate=0.04,
    currency=qs.Currency.USD,
    market_index=qs.MarketIndex.SOFR,
)
caplet = qs.CapletFloorlet(
    identifier="CAPLET",
    fixing_date=ref + "2Y",
    start_accrual_date=ref + "2Y",
    end_accrual_date=ref + "2Y3M",
    payment_date=ref + "2Y3M",
    strike=0.045,
    currency=qs.Currency.USD,
    market_index=qs.MarketIndex.SOFR,
    payoff_type=qs.CapletFloorletType.Caplet,
)
swap_trade = qs.Trade(swap, ref)
caplet_trade = qs.Trade(caplet, ref, notional=1_000_000)
rate_up = [
    qs.Scenario(
        label,
        0.001,
        qs.ScenarioType.Absolute,
    )
    for label in curve_spec["quotes"]
]
vol_up = [
    qs.Scenario(
        label,
        0.01,
        qs.ScenarioType.Absolute,
    )
    for label in surface_spec["quotes"]
]


def observe_market(scenarios):
    context = qs.PricingContext(
        quotes=market.quotes,
        curves=[curve_config],
        csa_policy=market.csa_policy,
        volatility_surfaces=[surface_config],
        scenarios=scenarios,
    )

    with context:
        return {
            "swap_pv_usd": context.evaluate(swap_trade).price,
            "caplet_pv_usd": context.evaluate(caplet_trade).price,
            "five_year_df": context.curve(qs.MarketIndex.SOFR).discount_factor(
                ref + "5Y"
            ),
            "two_year_vol": context.volatility_surface(qs.MarketIndex.SOFR).volatility(
                "2Y",
                0.045,
            ),
        }


comparison = pd.DataFrame(
    {
        "base": observe_market([]),
        "rates +10 bp": observe_market(rate_up),
        "vols +1 point": observe_market(vol_up),
    }
).T
comparison["swap_pnl_usd"] = (
    comparison["swap_pv_usd"] - comparison.loc["base", "swap_pv_usd"]
)
comparison["caplet_pnl_usd"] = (
    comparison["caplet_pv_usd"] - comparison.loc["base", "caplet_pv_usd"]
)
comparison

For this receive-fixed swap, higher rate quotes reduce PV. A volatility-only
move leaves its value unchanged because its pricer does not use a volatility
surface. Higher Black volatility increases the long caplet's option value.
The caplet's response to rates combines a change in its forward with a change
in discounting; do not interpret it as a pure discount-factor effect.

The table and plot show the economic effects of each market shock.

In [ ]:
comparison[["swap_pnl_usd", "caplet_pnl_usd"]].iloc[1:].plot.bar(
    title="Full revaluation under independent market shocks",
    ylabel="USD P&L",
    rot=0,
)
plt.axhline(
    0,
    color="black",
    linewidth=0.7,
)
plt.tight_layout()
plt.show()


For local risk, [portfolio_risk.ipynb](../sensitivity/portfolio_risk.ipynb) compares quote
derivatives with small full-revaluation bumps. For callable bonds or simulated
rates, the surface instead calibrates a short-rate model: its Black quotes are
not the model's absolute short-rate sigma. Follow that extra construction step
in [simulations](../hullwhite/simulations.ipynb) and [stepped bonds](../valuation/bonds/stepped_coupon_bond.ipynb).